In [2]:
import pandas as pd

data analysis

In [3]:
df = pd.read_csv("food_waste_management_dataset.csv")

In [4]:
df.head

<bound method NDFrame.head of            date day_of_week         menu  expected_students  actual_students  \
0    2026-01-01    Thursday  Paneer Rice                485              481   
1    2026-01-02      Friday  Idli Sambar                444              436   
2    2026-01-03    Saturday   Rajma Rice                373              336   
3    2026-01-04      Sunday   Rajma Rice                135              115   
4    2026-01-05      Monday  Paneer Rice                184              171   
..          ...         ...          ...                ...              ...   
295  2026-10-23      Friday  Paneer Rice                500              500   
296  2026-10-24    Saturday         Poha                362              347   
297  2026-10-25      Sunday    Veg Pulao                274              229   
298  2026-10-26      Monday  Paneer Rice                155              143   
299  2026-10-27     Tuesday   Rajma Rice                478              478   

     food

In [28]:
df.describe()
df.shape

(300, 18)

In [29]:
df.isna().sum() # no missing values 

date                         0
day_of_week                  0
menu                         0
expected_students            0
actual_students              0
food_prepared_portions       0
food_consumed_portions       0
food_wasted_portions         0
weather                      0
temperature_c                0
is_holiday                   0
is_exam_day                  0
special_event                0
consumption_rate             0
waste_percentage             0
waste_kg_est                 0
food_cost_per_portion_inr    0
waste_cost_inr               0
dtype: int64

In [30]:
print(df['food_prepared_portions'].mean())
print(df['food_consumed_portions'].mean())
print(df['food_wasted_portions'].mean())

403.07
340.4166666666667
62.653333333333336


62.65 portions are wasted per meal/day on average in our dataset.

In [ ]:
# particular food wise wastage

print(
    df.groupby('menu')['food_wasted_portions']
    .mean()
    .sort_values(ascending=False)
)

menu
Khichdi          101.896552
Poha              80.916667
Veg Pulao         72.378378
Rajma Rice        66.575758
Idli Sambar       65.300000
Dal Rice          56.666667
Dal Roti          50.571429
Chole Bhature     43.655172
Paneer Rice       40.481481
Pav Bhaji         37.280000
Name: food_wasted_portions, dtype: float64


In [ ]:
#  waste in percentage

print(
    df.groupby('menu')['waste_percentage']
    .mean()
    .sort_values(ascending=False)
)

menu
Khichdi          23.178621
Poha             20.940833
Veg Pulao        18.534595
Rajma Rice       17.015455
Idli Sambar      16.588500
Dal Rice         13.631944
Dal Roti         13.498571
Chole Bhature    11.787586
Paneer Rice      11.366296
Pav Bhaji         8.609200
Name: waste_percentage, dtype: float64


In [ ]:
#  consumption is greater than actual students
#  this is problametic 
#  in these model we have assume that 1 portion = one student meal 
print(
    df[df['food_consumed_portions'] > df['actual_students']]
)

           date day_of_week           menu  expected_students  \
4    2026-01-05      Monday    Paneer Rice                184   
45   2026-02-15      Sunday       Dal Roti                211   
53   2026-02-23      Monday      Pav Bhaji                469   
121  2026-05-02    Saturday  Chole Bhature                367   
234  2026-08-23      Sunday    Paneer Rice                209   

     actual_students  food_prepared_portions  food_consumed_portions  \
4                171                     193                     178   
45               211                     218                     214   
53               461                     477                     466   
121              339                     369                     345   
234              209                     216                     216   

     food_wasted_portions weather  temperature_c  is_holiday  is_exam_day  \
4                      15  Normal           30.9           1            0   
45                    

In [36]:
print(
    df[df['food_consumed_portions'] > df['food_prepared_portions']]
)

Empty DataFrame
Columns: [date, day_of_week, menu, expected_students, actual_students, food_prepared_portions, food_consumed_portions, food_wasted_portions, weather, temperature_c, is_holiday, is_exam_day, special_event, consumption_rate, waste_percentage, waste_kg_est, food_cost_per_portion_inr, waste_cost_inr]
Index: []


In [ ]:
#  this are the number of invalid records

invalid_records = df[
    df['food_consumed_portions'] > df['actual_students']
]

print("Invalid records:", len(invalid_records))
print("Percentage:", len(invalid_records) / len(df) * 100)

Invalid records: 5
Percentage: 1.6666666666666667


In [ ]:
#  this are the invalid records
print(
    invalid_records[
        ['date', 'menu', 'actual_students',
         'food_prepared_portions',
         'food_consumed_portions',
         'food_wasted_portions']
    ].to_string(index=False)
)

      date          menu  actual_students  food_prepared_portions  food_consumed_portions  food_wasted_portions
2026-01-05   Paneer Rice              171                     193                     178                    15
2026-02-15      Dal Roti              211                     218                     214                     4
2026-02-23     Pav Bhaji              461                     477                     466                    11
2026-05-02 Chole Bhature              339                     369                     345                    24
2026-08-23   Paneer Rice              209                     216                     216                     0


In [39]:
# Find invalid records
invalid = df['food_consumed_portions'] > df['actual_students']

# Correct consumed portions
df.loc[invalid, 'food_consumed_portions'] = (
    df.loc[invalid, 'actual_students']
)

# Recalculate dependent columns
df['food_wasted_portions'] = (
    df['food_prepared_portions']
    - df['food_consumed_portions']
)

df['consumption_rate'] = (
    df['food_consumed_portions']
    / df['actual_students']
).round(3)

df['waste_percentage'] = (
    df['food_wasted_portions']
    / df['food_prepared_portions']
    * 100
).round(2)

df['waste_kg_est'] = (
    df['food_wasted_portions'] * 0.35
).round(2)

df['waste_cost_inr'] = (
    df['food_wasted_portions']
    * df['food_cost_per_portion_inr']
).round(2)

print("Remaining invalid records:")

print(
    df[
        df['food_consumed_portions']
        > df['actual_students']
    ]
)

Remaining invalid records:
Empty DataFrame
Columns: [date, day_of_week, menu, expected_students, actual_students, food_prepared_portions, food_consumed_portions, food_wasted_portions, weather, temperature_c, is_holiday, is_exam_day, special_event, consumption_rate, waste_percentage, waste_kg_est, food_cost_per_portion_inr, waste_cost_inr]
Index: []


we get the cleaned file where the invalid records are removed

In [43]:
df.to_csv(
    "food_waste_management_dataset_cleaned.csv",
    index=False
)